# 02 Dataset cleaning and joining
Inputs: FIPS reference (`data/`), IM3 Atlas and projected scenarios (`data/im3/`), NCSL bills (`data/ncsl_ai/output/ncsl_ai_db_final.csv`), and `cerf_county_tabular.csv` from notebook 01. Outputs: `data/cerf_siting/output/stage2_clean_join/`.

Run this notebook in VS Code Jupyter with the repository open. The setup locates the repository from the Jupyter working directory or `REPO_DIR_OVERRIDE`.


In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np

from pathlib import Path
import subprocess

In [3]:
# Notebooks live in scripts/; data lives in data/ (CERF rasters and outputs in data/cerf_siting/, NCSL in data/ncsl_ai/).
REPO_DIR_OVERRIDE = None  # Optional: full local repository path if Jupyter runs elsewhere

def find_repo():
    if REPO_DIR_OVERRIDE is not None:
        root = Path(REPO_DIR_OVERRIDE).expanduser().resolve()
    else:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents]
        root = next((p for p in candidates if (p / 'data' / 'cerf_siting').is_dir()), None)
        if root is None:
            result = subprocess.run(['git', 'rev-parse', '--show-toplevel'], cwd=cwd, capture_output=True, text=True)
            if result.returncode == 0:
                root = Path(result.stdout.strip()).resolve()
        if root is None:
            raise FileNotFoundError('Jupyter is outside the repository. Open the notebook in VS Code from the repo, or set REPO_DIR_OVERRIDE.')
    if not (root / 'data' / 'cerf_siting').is_dir():
        raise FileNotFoundError(f'Expected data/cerf_siting/ under {root}')
    return root

project_dir = find_repo()
DATA_ROOT = project_dir / 'data'
CERF_RASTER_DIR = DATA_ROOT / 'cerf_siting' / 'raw_data'                  # CERF .tif rasters
NCSL_OUTPUT_DIR = DATA_ROOT / 'ncsl_ai' / 'output'                         # NCSL pipeline outputs
OUTPUT_DIR = DATA_ROOT / 'cerf_siting' / 'output'                          # shared stage outputs: stage1_..., stage2_..., ...
RUN_DIR = DATA_ROOT                                                       # find_input() searches here
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_input(filename=None, folder=None, search_dir=None):
    base = search_dir or RUN_DIR
    if (filename is None) == (folder is None):
        raise ValueError('Specify exactly one of filename or folder')
    matches = sorted(p for p in base.rglob('*') if (p.is_file() and p.name.lower() == filename.lower()) if filename is not None) if filename else sorted(p for p in base.rglob('*') if p.is_dir() and p.name.lower() == folder.lower())
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {filename or folder} under {base}; found {len(matches)}: {matches[:10]}')
    return matches[0]

print('Repository:', project_dir.name)
print('CERF rasters:', CERF_RASTER_DIR.relative_to(project_dir).as_posix())
print('Cleaning inputs:', RUN_DIR.relative_to(project_dir).as_posix())
print('Stage outputs:', OUTPUT_DIR.relative_to(project_dir).as_posix())

Repository: ai-law-datacenter-siting
CERF rasters: data/cerf_siting/raw_data
Cleaning inputs: data
Stage outputs: data/cerf_siting/output


In [4]:
output_dir = OUTPUT_DIR / 'stage2_clean_join'
output_dir.mkdir(parents=True, exist_ok=True)

quality_dir = OUTPUT_DIR / 'quality_checks'
quality_dir.mkdir(parents=True, exist_ok=True)

cerf_path = OUTPUT_DIR / 'stage1_cerf_extraction' / 'cerf_county_tabular.csv'


fips_path = find_input(
    filename='state_county_and_city_fips_reference_table.csv'
)

ncsl_path = NCSL_OUTPUT_DIR / 'ncsl_ai_db_final.csv'
if not ncsl_path.is_file():
    raise FileNotFoundError(f'NCSL file missing: {ncsl_path}')
im3_dir = find_input(folder='projected_data_centers')
atlas_path = find_input(filename='im3_data_center_atlas_v2026.02.09.csv')
for name, p in {'CERF':cerf_path,'FIPS':fips_path,'NCSL':ncsl_path,'IM3':im3_dir,'Atlas':atlas_path}.items():
    print(name, p.relative_to(project_dir).as_posix(), 'EXISTS' if p.exists() else 'MISSING')
if not cerf_path.exists():
    raise FileNotFoundError('Run 01_CERF_Raster_Extraction.ipynb first to create the county CSV.')

CERF data/cerf_siting/output/stage1_cerf_extraction/cerf_county_tabular.csv EXISTS
FIPS data/state_county_and_city_fips_reference_table.csv EXISTS
NCSL data/ncsl_ai/output/ncsl_ai_db_final.csv EXISTS
IM3 data/im3/projected_data_centers EXISTS
Atlas data/im3/im3_data_center_atlas_v2026.02.09.csv EXISTS


# Find all IM3 GeoJSON files


In [11]:
# Locate IM3 projection files

im3_files = sorted(im3_dir.rglob('*.geojson'))

print(f'IM3 GeoJSON files found: {len(im3_files)}')

for file in im3_files:
    print(file.relative_to(im3_dir))

IM3 GeoJSON files found: 20
high_growth\high_growth_0_market_gravity.geojson
high_growth\high_growth_100_market_gravity.geojson
high_growth\high_growth_25_market_gravity.geojson
high_growth\high_growth_50_market_gravity.geojson
high_growth\high_growth_75_market_gravity.geojson
higher_growth\higher_growth_0_market_gravity.geojson
higher_growth\higher_growth_100_market_gravity.geojson
higher_growth\higher_growth_25_market_gravity.geojson
higher_growth\higher_growth_50_market_gravity.geojson
higher_growth\higher_growth_75_market_gravity.geojson
low_growth\low_growth_0_market_gravity.geojson
low_growth\low_growth_100_market_gravity.geojson
low_growth\low_growth_25_market_gravity.geojson
low_growth\low_growth_50_market_gravity.geojson
low_growth\low_growth_75_market_gravity.geojson
moderate_growth\moderate_growth_0_market_gravity.geojson
moderate_growth\moderate_growth_100_market_gravity.geojson
moderate_growth\moderate_growth_25_market_gravity.geojson
moderate_growth\moderate_growth_50_mar

Convert GeoJSON files to CSV

In [12]:
# Folder for converted IM3 datasets
im3_output_dir = output_dir / 'im3_csv'
im3_output_dir.mkdir(parents=True, exist_ok=True)

converted_files = []

for file in im3_files:

    # Read GeoJSON
    gdf = gpd.read_file(file)

    # Preserve scenario information
    gdf['growth_scenario'] = file.parent.name
    gdf['market_gravity'] = file.stem.split('_market_gravity')[0].split('_')[-1]

    # Store geometry as text so spatial information is not lost
    gdf['geometry_wkt'] = gdf.geometry.to_wkt()

    # Convert to regular DataFrame
    df = pd.DataFrame(gdf.drop(columns='geometry'))

    # Save as CSV
    output_file = im3_output_dir / f'{file.stem}.csv'
    df.to_csv(output_file, index=False)

    converted_files.append({
        'scenario': file.parent.name,
        'file': output_file.name,
        'rows': len(df),
        'columns': len(df.columns)
    })

    print(f'Converted: {output_file.name}')

Converted: high_growth_0_market_gravity.csv
Converted: high_growth_100_market_gravity.csv
Converted: high_growth_25_market_gravity.csv
Converted: high_growth_50_market_gravity.csv
Converted: high_growth_75_market_gravity.csv
Converted: higher_growth_0_market_gravity.csv
Converted: higher_growth_100_market_gravity.csv
Converted: higher_growth_25_market_gravity.csv
Converted: higher_growth_50_market_gravity.csv
Converted: higher_growth_75_market_gravity.csv
Converted: low_growth_0_market_gravity.csv
Converted: low_growth_100_market_gravity.csv
Converted: low_growth_25_market_gravity.csv
Converted: low_growth_50_market_gravity.csv
Converted: low_growth_75_market_gravity.csv
Converted: moderate_growth_0_market_gravity.csv
Converted: moderate_growth_100_market_gravity.csv
Converted: moderate_growth_25_market_gravity.csv
Converted: moderate_growth_50_market_gravity.csv
Converted: moderate_growth_75_market_gravity.csv


Review converted datasets

In [13]:
# Summary of converted IM3 files

conversion_summary = pd.DataFrame(converted_files)

display(conversion_summary)

print(f'Total CSV files: {len(conversion_summary)}')
print(f'Total records: {conversion_summary["rows"].sum():,}')

conversion_summary.to_csv(
    im3_output_dir / 'conversion_summary.csv',
    index=False
)

,scenario,file,rows,columns
0,high_growth,high_growth_0_market_gravity.csv,909,17
1,high_growth,high_growth_100_market_gravity.csv,909,17
2,high_growth,high_growth_25_market_gravity.csv,909,17
3,high_growth,high_growth_50_market_gravity.csv,909,17
4,high_growth,high_growth_75_market_gravity.csv,909,17
5,higher_growth,higher_growth_0_market_gravity.csv,1870,17
6,higher_growth,higher_growth_100_market_gravity.csv,1870,17
7,higher_growth,higher_growth_25_market_gravity.csv,1870,17
8,higher_growth,higher_growth_50_market_gravity.csv,1870,17
9,higher_growth,higher_growth_75_market_gravity.csv,1870,17


Total CSV files: 20
Total records: 16,645


Preview and check each CSV

In [14]:
# Basic data quality checks

quality_checks = []

for file in sorted(im3_output_dir.glob('*.csv')):

    if file.name == 'conversion_summary.csv':
        continue

    df = pd.read_csv(file)

    quality_checks.append({
        'file': file.name,
        'rows': len(df),
        'columns': len(df.columns),
        'duplicates': int(df.duplicated().sum()),
        'missing_values': int(df.isna().sum().sum()),
        'missing_percent': round(
            df.isna().sum().sum() / df.size * 100, 2
        ) if df.size else 0
    })

quality_summary = pd.DataFrame(quality_checks)

display(quality_summary)

quality_summary.to_csv(
    im3_output_dir / 'quality_summary.csv',
    index=False
)

,file,rows,columns,duplicates,missing_values,missing_percent
0,high_growth_0_market_gravity.csv,909,17,0,0,0.0
1,high_growth_100_market_gravity.csv,909,17,0,0,0.0
2,high_growth_25_market_gravity.csv,909,17,0,0,0.0
3,high_growth_50_market_gravity.csv,909,17,0,0,0.0
4,high_growth_75_market_gravity.csv,909,17,0,0,0.0
5,higher_growth_0_market_gravity.csv,1870,17,0,0,0.0
6,higher_growth_100_market_gravity.csv,1870,17,0,0,0.0
7,higher_growth_25_market_gravity.csv,1870,17,0,0,0.0
8,higher_growth_50_market_gravity.csv,1870,17,0,0,0.0
9,higher_growth_75_market_gravity.csv,1870,17,0,0,0.0


Load the datasets

In [15]:
# Load datasets

cerf = pd.read_csv(cerf_path, dtype={'county_fips': 'string'})
fips_ref = pd.read_csv(fips_path, dtype='string')
ncsl = pd.read_csv(ncsl_path, dtype='string')
atlas = pd.read_csv(atlas_path, dtype='string')

# Organize datasets
datasets = {
    'CERF': cerf,
    'FIPS': fips_ref,
    'NCSL': ncsl,
    'IM3 Atlas': atlas
}

# Check dataset sizes
for name, df in datasets.items():
    df.columns = df.columns.str.strip()
    print(f'{name}: {df.shape[0]:,} rows, {df.shape[1]} columns')

CERF: 3,235 rows, 31 columns
FIPS: 40,411 rows, 9 columns
NCSL: 1,907 rows, 73 columns
IM3 Atlas: 1,479 rows, 13 columns


We'll use a small number of reusable functions rather than repeating the same code for every dataset.

In [16]:
def clean_fips(series, length):
    """Format geographic identifiers while preserving leading zeros."""
    values = series.astype('string').str.strip()
    values = values.str.replace(r'\.0$', '', regex=True)

    valid = values.str.fullmatch(rf'\d{{1,{length}}}', na=False)

    return values.where(valid).str.zfill(length)


def check_duplicates(df, column, name):
    duplicates = df[df.duplicated(column, keep=False)]

    if not duplicates.empty:
        duplicates.to_csv(
            output_dir / f'{name}_duplicate_keys.csv',
            index=False
        )
        print(f'{name}: {len(duplicates):,} rows with duplicate keys')

    return duplicates


def save_clean(df, filename):
    df.to_csv(output_dir / filename, index=False)
    print(f'Saved {filename}: {len(df):,} rows')

Our FIPS reference includes multiple cities within the same county. We only need one record per county.

In [17]:
fips_clean = fips_ref[
    ['StCnty FIPS Code', 'State FIPS Code',
     'State Code', 'State Name', 'County Name']
].copy()

fips_clean.columns = [
    'county_fips', 'state_fips',
    'state_abbr', 'state_name', 'county_name'
]

fips_clean['county_fips'] = clean_fips(fips_clean['county_fips'], 5)
fips_clean = fips_clean.dropna(subset=['county_fips'])

fips_clean['state_fips'] = fips_clean['county_fips'].str[:2]

# Check whether repeated county rows disagree on county information
county_conflicts = (
    fips_clean.groupby('county_fips')
    [['state_abbr', 'state_name', 'county_name']]
    .nunique(dropna=True)
)

county_conflicts = county_conflicts[
    county_conflicts.gt(1).any(axis=1)
]

county_conflicts.to_csv(output_dir / 'fips_conflicts.csv')

# Repeated city rows are expected in this reference
fips_clean = fips_clean.drop_duplicates('county_fips')

save_clean(fips_clean, '01_fips_county_clean.csv')

print('Conflicting county identifiers:', len(county_conflicts))
fips_clean.head()

Saved 01_fips_county_clean.csv: 3,154 rows
Conflicting county identifiers: 0


,county_fips,state_fips,state_abbr,state_name,county_name
0,01001,01,AL,ALABAMA,AUTAUGA
6,01003,01,AL,ALABAMA,BALDWIN
30,01005,01,AL,ALABAMA,BARBOUR
38,01007,01,AL,ALABAMA,BIBB
47,01009,01,AL,ALABAMA,BLOUNT


We're using your already aggregated county-level CERF file here. This cell cleans it without aggregating it again or removing its existing statistical features.

In [18]:
cerf_clean = cerf.copy()

cerf_clean['county_fips'] = clean_fips(
    cerf_clean['county_fips'], 5
)

# Remove rows without a valid county identifier
cerf_invalid = cerf_clean[cerf_clean['county_fips'].isna()]
cerf_invalid.to_csv(output_dir / 'cerf_invalid_fips.csv', index=False)

cerf_clean = cerf_clean.dropna(subset=['county_fips'])

# County names and geographic identifiers come from the FIPS reference
extra_columns = ['STATEFP', 'COUNTYFP', 'county_name', 'NAMELSAD']
cerf_clean = cerf_clean.drop(
    columns=extra_columns,
    errors='ignore'
)

# CERF feature columns should be numeric
for column in cerf_clean.columns:
    if column != 'county_fips':
        cerf_clean[column] = pd.to_numeric(
            cerf_clean[column], errors='coerce'
        )

cerf_clean = cerf_clean.replace([np.inf, -np.inf], np.nan)

cerf_duplicates = check_duplicates(
    cerf_clean, 'county_fips', 'cerf'
)

if not cerf_duplicates.empty:
    raise ValueError('Review duplicate CERF counties before joining.')

save_clean(cerf_clean, '02_cerf_county_clean.csv')

cerf_clean.head()

Saved 02_cerf_county_clean.csv: 3,235 rows


,county_fips,centroid_longitude,centroid_latitude,commercial_electricity_rates_dollar_per_kwh_mean,commercial_electricity_rates_dollar_per_kwh_median,composite_siting_suitability_mean,composite_siting_suitability_median,composite_siting_suitability_max,data_center_market_locations_majority,distance_to_substation_mean,...,mechanical_cooling_fraction_median,personal_property_tax_rate_mean,personal_property_tax_rate_median,real_property_tax_rate_mean,real_property_tax_rate_median,sales_tax_rate_mean,sales_tax_rate_median,state_siting_region_majority,water_cooling_fraction_mean,water_cooling_fraction_median
0,31039,-96.787400,41.916312,0.105964,0.105964,0.005382,0.0,1.0,0.0,103.055201,...,1.0,0.0125,0.0125,0.010377,0.010377,0.069723,0.069723,31.0,0.000000,0.0
1,53069,-123.433359,46.291139,0.063310,0.063310,0.004687,0.0,1.0,0.0,87.761017,...,0.0,0.0125,0.0125,0.006049,0.006049,0.094116,0.094293,53.0,1.000000,1.0
2,35011,-104.412089,34.342297,0.078584,0.078584,0.001105,0.0,1.0,0.0,314.773379,...,1.0,0.0000,0.0000,0.004987,0.004987,0.076270,0.076270,35.0,0.000000,0.0
3,31109,-96.687757,40.783894,0.102269,0.102269,0.026213,0.0,1.0,0.0,56.637924,...,0.0,0.0125,0.0125,0.015662,0.015662,0.069723,0.069723,31.0,0.986992,1.0
4,31129,-98.047185,40.176298,0.207833,0.207833,0.000000,0.0,0.0,0.0,317.405566,...,1.0,0.0125,0.0125,0.012805,0.012805,0.069724,0.069723,31.0,0.087278,0.0


These two datasets need different transformations: IM3 becomes a county-level count, while NCSL becomes a state-level summary.

**Atlas schema check:** The original code below expects `state_id`, `county_id`, and `id`. If your Atlas uses different columns, inspect the output before running its aggregation.

In [19]:
print('Atlas columns:', atlas.columns.tolist())
required = {'state_id','county_id','id'}
if not required.issubset(atlas.columns):
    raise KeyError(f'Atlas is missing original expected columns: {sorted(required - set(atlas.columns))}. Inspect Atlas schema before aggregation.')

Atlas columns: ['id', 'state', 'state_abb', 'state_id', 'county', 'county_id', 'operator', 'ref', 'name', 'sqft', 'lon', 'lat', 'type']


In [20]:
# IM3 Atlas: facility records to county counts
atlas_clean = atlas.copy()

atlas_clean['state_fips'] = clean_fips(atlas_clean['state_id'], 2)
atlas_clean['county_code'] = clean_fips(atlas_clean['county_id'], 3)

atlas_clean['county_fips'] = (
    atlas_clean['state_fips'] + atlas_clean['county_code']
)

# Keep repeated facility IDs for review rather than deleting them
atlas_clean[atlas_clean.duplicated('id', keep=False)].to_csv(
    output_dir / 'atlas_repeated_ids.csv', index=False
)

atlas_clean[
    atlas_clean['county_fips'].isna() |
    ~atlas_clean['county_fips'].isin(fips_clean['county_fips'])
].to_csv(output_dir / 'atlas_unmatched_counties.csv', index=False)


# Exclude facility IDs with conflicting county assignments
# until their locations can be verified

conflicting_ids = (
    atlas_clean.groupby("id")["county_fips"]
    .nunique()
)

conflicting_ids = conflicting_ids[
    conflicting_ids > 1
].index

atlas_review = atlas_clean[
    atlas_clean["id"].isin(conflicting_ids)
].copy()

atlas_review.to_csv(
    output_dir / "atlas_conflicting_counties.csv",
    index=False
)

atlas_unambiguous = atlas_clean[
    ~atlas_clean["id"].isin(conflicting_ids)
].copy()

atlas_county = (
    atlas_unambiguous.dropna(subset=["county_fips"])
    .groupby("county_fips")
    .size()
    .reset_index(name="atlas_record_count")
)

print("Conflicting facility IDs:", len(conflicting_ids))
print("Records excluded for review:", len(atlas_review))


atlas_county['has_existing_data_center'] = 1

save_clean(atlas_county, '03_atlas_county_clean.csv')

Conflicting facility IDs: 5
Records excluded for review: 10
Saved 03_atlas_county_clean.csv: 250 rows


In [21]:
# Save processed Atlas data and records requiring review
atlas_county.to_csv(
    output_dir / "03_atlas_county_clean.csv",
    index=False
)

atlas_review.to_csv(
    output_dir / "atlas_conflicting_counties.csv",
    index=False
)

print("Processed files saved:")
print(output_dir / "03_atlas_county_clean.csv")
print(output_dir / "atlas_conflicting_counties.csv")


Processed files saved:
C:\Users\chals\Downloads\ai-law-datacenter-siting\data\cerf_siting\output\stage2_clean_join\03_atlas_county_clean.csv
C:\Users\chals\Downloads\ai-law-datacenter-siting\data\cerf_siting\output\stage2_clean_join\atlas_conflicting_counties.csv


In [22]:

# Review repeated Atlas facility IDs

repeated_ids = atlas_clean[
    atlas_clean.duplicated("id", keep=False)
].copy()

print("Records with repeated IDs:", len(repeated_ids))
print("Unique IDs repeated:", repeated_ids["id"].nunique())

if not repeated_ids.empty:
    print("\nRepeated facility records:")
    display(
        repeated_ids[
            ["id", "county_fips"]
        ].sort_values("id").head(20)
    )

    print("\nIDs associated with multiple counties:")
    county_counts = (
        repeated_ids.groupby("id")["county_fips"]
        .nunique()
    )

    print((county_counts > 1).sum())


Records with repeated IDs: 10
Unique IDs repeated: 5

Repeated facility records:


,id,county_fips
735,00830885779,19121
736,00830885779,19181
1360,00844391357,39049
1361,00844391357,39089
829,00975064000,13097
830,00975064000,13067
977,01126774147,29047
978,01126774147,29165
1239,01386016633,39049
1240,01386016633,39089



IDs associated with multiple counties:
5


In [25]:
# NCSL: legislation records to state-level policy features

ncsl_clean = ncsl.copy()

# Map state abbreviations to two digit FIPS codes
state_abbrev_to_fips = {
    'AL':'01', 'AK':'02', 'AZ':'04', 'AR':'05', 'CA':'06',
    'CO':'08', 'CT':'09', 'DE':'10', 'DC':'11', 'FL':'12',
    'GA':'13', 'HI':'15', 'ID':'16', 'IL':'17', 'IN':'18',
    'IA':'19', 'KS':'20', 'KY':'21', 'LA':'22', 'ME':'23',
    'MD':'24', 'MA':'25', 'MI':'26', 'MN':'27', 'MS':'28',
    'MO':'29', 'MT':'30', 'NE':'31', 'NV':'32', 'NH':'33',
    'NJ':'34', 'NM':'35', 'NY':'36', 'NC':'37', 'ND':'38',
    'OH':'39', 'OK':'40', 'OR':'41', 'PA':'42', 'RI':'44',
    'SC':'45', 'SD':'46', 'TN':'47', 'TX':'48', 'UT':'49',
    'VT':'50', 'VA':'51', 'WA':'53', 'WV':'54', 'WI':'55',
    'WY':'56', 'PR':'72', 'GU':'66', 'VI':'78'
}

ncsl_clean['state_key'] = (
    ncsl_clean['state'].astype('string').str.upper().str.strip()
)

ncsl_clean['state_fips'] = (
    ncsl_clean['state_key'].map(state_abbrev_to_fips)
)

ncsl_clean['session_year'] = pd.to_numeric(
    ncsl_clean['session_year'], errors='coerce'
)

ncsl_clean['status_clean'] = (
    ncsl_clean['status']
    .fillna('')
    .str.lower()
    .str.replace('–', '-', regex=False)
    .str.strip()
)

ncsl_clean['is_enacted'] = (
    ncsl_clean['status_clean'].isin(['enacted', 'adopted'])
).astype(int)

ncsl_clean['bill_key'] = (
    ncsl_clean['bill_number'].fillna('').astype(str).str.upper().str.strip()
    + '_'
    + ncsl_clean['session_year'].astype('Int64').astype('string').fillna('')
)

ncsl_clean = ncsl_clean[
    ncsl_clean['bill_number'].notna()
].copy()

# Save records that did not match a state
ncsl_clean[ncsl_clean['state_fips'].isna()].to_csv(
    output_dir / 'ncsl_unmatched_states.csv', index=False
)

# Count each bill once per state and year
unique_bills = (
    ncsl_clean.dropna(subset=['state_fips'])
    .drop_duplicates(['state_fips', 'bill_key'])
)

ncsl_state = (
    unique_bills.groupby('state_fips')
    .agg(
        ncsl_bill_count=('bill_key', 'size'),
        ncsl_enacted_count=('is_enacted', 'sum')
    )
    .reset_index()
)

save_clean(ncsl_state, '04_ncsl_state_clean.csv')

print("Original NCSL records:", len(ncsl))
print("Matched records:", ncsl_clean['state_fips'].notna().sum())
print("Unmatched records:", ncsl_clean['state_fips'].isna().sum())
print("States and territories represented:", len(ncsl_state))

display(ncsl_state.head())

Saved 04_ncsl_state_clean.csv: 54 rows
Original NCSL records: 1907
Matched records: 1907
Unmatched records: 0
States and territories represented: 54


,state_fips,ncsl_bill_count,ncsl_enacted_count
0,01,17,8
1,02,11,1
2,04,15,4
3,05,15,6
4,06,123,12


This produces the master dataset that we'll use for EDA and modeling.

In [26]:
master = fips_clean.merge(
    cerf_clean,
    on='county_fips',
    how='left',
    validate='one_to_one',
    indicator='cerf_join'
)

master = master.merge(
    atlas_county,
    on='county_fips',
    how='left',
    validate='one_to_one',
    indicator='atlas_join'
)

master = master.merge(
    ncsl_state,
    on='state_fips',
    how='left',
    validate='many_to_one',
    indicator='ncsl_join'
)

# A missing Atlas record is not automatically a verified negative label
master['atlas_record_count'] = (
    master['atlas_record_count'].fillna(0).astype(int)
)

# Identify counties with at least one recorded Atlas data center
master['has_existing_data_center'] = (
    master['atlas_record_count'] > 0
).astype(int)

print("Atlas recorded presence:")
print(master['has_existing_data_center'].value_counts())

# Check join coverage before modeling
for source in ['cerf', 'atlas', 'ncsl']:
    column = f'{source}_join'
    matched = master[column].eq('both').sum()

    print(f'{source.upper()}: {matched:,} / {len(master):,} matched')

print(f'\nMaster dataset: {master.shape[0]:,} rows')
print(f'Total columns: {master.shape[1]}')

if master['county_fips'].duplicated().any():
    raise ValueError('The master dataset contains duplicate counties.')

save_clean(master, 'master_county_clean.csv')

master.head()

Atlas recorded presence:
has_existing_data_center
0    2910
1     244
Name: count, dtype: int64
CERF: 3,134 / 3,154 matched
ATLAS: 244 / 3,154 matched
NCSL: 3,153 / 3,154 matched

Master dataset: 3,154 rows
Total columns: 38
Saved master_county_clean.csv: 3,154 rows


,county_fips,state_fips,state_abbr,state_name,county_name,centroid_longitude,centroid_latitude,commercial_electricity_rates_dollar_per_kwh_mean,commercial_electricity_rates_dollar_per_kwh_median,composite_siting_suitability_mean,...,state_siting_region_majority,water_cooling_fraction_mean,water_cooling_fraction_median,cerf_join,atlas_record_count,has_existing_data_center,atlas_join,ncsl_bill_count,ncsl_enacted_count,ncsl_join
0,01001,01,AL,ALABAMA,AUTAUGA,-86.642761,32.534912,0.056590,0.056590,0.019192,...,1.0,0.807202,1.0,both,0,0,left_only,17.0,8.0,both
1,01003,01,AL,ALABAMA,BALDWIN,-87.749781,30.660276,0.058395,0.058395,0.021779,...,1.0,0.000000,0.0,both,0,0,left_only,17.0,8.0,both
2,01005,01,AL,ALABAMA,BARBOUR,-85.393431,31.869549,0.054066,0.054066,0.006813,...,1.0,0.108687,0.0,both,0,0,left_only,17.0,8.0,both
3,01007,01,AL,ALABAMA,BIBB,-87.126463,32.998615,0.055336,0.055336,0.010149,...,1.0,1.000000,1.0,both,0,0,left_only,17.0,8.0,both
4,01009,01,AL,ALABAMA,BLOUNT,-86.567545,33.980817,0.053852,0.053852,0.019309,...,1.0,1.000000,1.0,both,0,0,left_only,17.0,8.0,both


This displays the first five records from each dataset, along with the number of rows and columns.

In [27]:
# Preview cleaned datasets

cleaned_datasets = {
    'FIPS': fips_clean,
    'CERF': cerf_clean,
    'IM3 Atlas': atlas_county,
    'NCSL': ncsl_state,
    'Master': master
}

for name, df in cleaned_datasets.items():
    print(f'\n{name.upper()} DATASET')
    print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')

    display(df.head(5))


FIPS DATASET
Rows: 3,154 | Columns: 5


,county_fips,state_fips,state_abbr,state_name,county_name
0,01001,01,AL,ALABAMA,AUTAUGA
6,01003,01,AL,ALABAMA,BALDWIN
30,01005,01,AL,ALABAMA,BARBOUR
38,01007,01,AL,ALABAMA,BIBB
47,01009,01,AL,ALABAMA,BLOUNT



CERF DATASET
Rows: 3,235 | Columns: 27


,county_fips,centroid_longitude,centroid_latitude,commercial_electricity_rates_dollar_per_kwh_mean,commercial_electricity_rates_dollar_per_kwh_median,composite_siting_suitability_mean,composite_siting_suitability_median,composite_siting_suitability_max,data_center_market_locations_majority,distance_to_substation_mean,...,mechanical_cooling_fraction_median,personal_property_tax_rate_mean,personal_property_tax_rate_median,real_property_tax_rate_mean,real_property_tax_rate_median,sales_tax_rate_mean,sales_tax_rate_median,state_siting_region_majority,water_cooling_fraction_mean,water_cooling_fraction_median
0,31039,-96.787400,41.916312,0.105964,0.105964,0.005382,0.0,1.0,0.0,103.055201,...,1.0,0.0125,0.0125,0.010377,0.010377,0.069723,0.069723,31.0,0.000000,0.0
1,53069,-123.433359,46.291139,0.063310,0.063310,0.004687,0.0,1.0,0.0,87.761017,...,0.0,0.0125,0.0125,0.006049,0.006049,0.094116,0.094293,53.0,1.000000,1.0
2,35011,-104.412089,34.342297,0.078584,0.078584,0.001105,0.0,1.0,0.0,314.773379,...,1.0,0.0000,0.0000,0.004987,0.004987,0.076270,0.076270,35.0,0.000000,0.0
3,31109,-96.687757,40.783894,0.102269,0.102269,0.026213,0.0,1.0,0.0,56.637924,...,0.0,0.0125,0.0125,0.015662,0.015662,0.069723,0.069723,31.0,0.986992,1.0
4,31129,-98.047185,40.176298,0.207833,0.207833,0.000000,0.0,0.0,0.0,317.405566,...,1.0,0.0125,0.0125,0.012805,0.012805,0.069724,0.069723,31.0,0.087278,0.0



IM3 ATLAS DATASET
Rows: 250 | Columns: 3


,county_fips,atlas_record_count,has_existing_data_center
0,01069,1,1
1,01071,3,1
2,01073,1,1
3,01089,6,1
4,04003,1,1



NCSL DATASET
Rows: 54 | Columns: 3


,state_fips,ncsl_bill_count,ncsl_enacted_count
0,01,17,8
1,02,11,1
2,04,15,4
3,05,15,6
4,06,123,12



MASTER DATASET
Rows: 3,154 | Columns: 38


,county_fips,state_fips,state_abbr,state_name,county_name,centroid_longitude,centroid_latitude,commercial_electricity_rates_dollar_per_kwh_mean,commercial_electricity_rates_dollar_per_kwh_median,composite_siting_suitability_mean,...,state_siting_region_majority,water_cooling_fraction_mean,water_cooling_fraction_median,cerf_join,atlas_record_count,has_existing_data_center,atlas_join,ncsl_bill_count,ncsl_enacted_count,ncsl_join
0,01001,01,AL,ALABAMA,AUTAUGA,-86.642761,32.534912,0.056590,0.056590,0.019192,...,1.0,0.807202,1.0,both,0,0,left_only,17.0,8.0,both
1,01003,01,AL,ALABAMA,BALDWIN,-87.749781,30.660276,0.058395,0.058395,0.021779,...,1.0,0.000000,0.0,both,0,0,left_only,17.0,8.0,both
2,01005,01,AL,ALABAMA,BARBOUR,-85.393431,31.869549,0.054066,0.054066,0.006813,...,1.0,0.108687,0.0,both,0,0,left_only,17.0,8.0,both
3,01007,01,AL,ALABAMA,BIBB,-87.126463,32.998615,0.055336,0.055336,0.010149,...,1.0,1.000000,1.0,both,0,0,left_only,17.0,8.0,both
4,01009,01,AL,ALABAMA,BLOUNT,-86.567545,33.980817,0.053852,0.053852,0.019309,...,1.0,1.000000,1.0,both,0,0,left_only,17.0,8.0,both


This checks record counts, duplicates, missing values, and column types.

In [28]:
# High-level data quality checks

quality_results = []

for name, df in cleaned_datasets.items():

    key = 'state_fips' if name == 'NCSL' else 'county_fips'

    quality_results.append({
        'Dataset': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Duplicate Rows': df.duplicated().sum(),
        'Duplicate Keys': df[key].duplicated().sum(),
        'Missing Values': df.isna().sum().sum(),
        'Missing %': round(
            df.isna().sum().sum() / df.size * 100, 2
        ) if df.size else 0,
        'Numeric Columns': len(df.select_dtypes(include='number').columns),
        'Text Columns': len(df.select_dtypes(include=['object', 'string']).columns)
    })

quality_summary = pd.DataFrame(quality_results)

display(quality_summary)


# Columns with the most missing values

for name, df in cleaned_datasets.items():

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print(f'\n{name.upper()} — MISSING VALUES')

    if missing.empty:
        print('No missing values found.')
    else:
        display(
            pd.DataFrame({
                'Missing Count': missing,
                'Missing %': (missing / len(df) * 100).round(2)
            }).head(15)
        )


# Master dataset join coverage

print('\nMASTER DATASET — JOIN COVERAGE')

for source in ['cerf', 'atlas', 'ncsl']:
    column = f'{source}_join'

    print(f'\n{source.upper()}')
    display(master[column].value_counts(dropna=False))

,Dataset,Rows,Columns,Duplicate Rows,Duplicate Keys,Missing Values,Missing %,Numeric Columns,Text Columns
0,FIPS,3154,5,0,0,2,0.01,0,5
1,CERF,3235,27,0,0,3024,3.46,26,1
2,IM3 Atlas,250,3,0,0,0,0.00,2,1
3,NCSL,54,3,0,0,0,0.00,2,1
4,Master,3154,38,0,0,1364,1.14,30,5



FIPS — MISSING VALUES


,Missing Count,Missing %
state_abbr,1,0.03
state_name,1,0.03



CERF — MISSING VALUES


,Missing Count,Missing %
commercial_electricity_rates_dollar_per_kwh_mean,126,3.89
commercial_electricity_rates_dollar_per_kwh_median,126,3.89
composite_siting_suitability_mean,126,3.89
composite_siting_suitability_median,126,3.89
composite_siting_suitability_max,126,3.89
data_center_market_locations_majority,126,3.89
distance_to_substation_mean,126,3.89
distance_to_substation_min,126,3.89
distance_to_substation_median,126,3.89
industrial_electricity_rates_dollar_per_kwh_mean,126,3.89


...

cerf_join
both          3134
left_only       20
right_only       0
Name: count, dtype: int64


ATLAS


atlas_join
left_only     2910
both           244
right_only       0
Name: count, dtype: int64


NCSL


ncsl_join
both          3153
left_only        1
right_only       0
Name: count, dtype: int64

In [ ]:
# Check missing values and dataset joins

print("Missing values by column:")
display(
    master.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Count")
)

print("\nJoin results:")
for col in master.columns:
    if col.endswith("_join"):
        print(f"\n{col}")
        print(master[col].value_counts(dropna=False))

print("\nDuplicate county FIPS:")
print(master["county_fips"].duplicated().sum())

Missing values by column:


,Missing Count
land_value_dollar_per_sqft_median,55
distance_to_substation_median,55
water_cooling_fraction_median,55
water_cooling_fraction_mean,55
state_siting_region_majority,55
sales_tax_rate_median,55
sales_tax_rate_mean,55
real_property_tax_rate_median,55
real_property_tax_rate_mean,55
personal_property_tax_rate_median,55



Join results:

cerf_join
cerf_join
both          3134
left_only       20
right_only       0
Name: count, dtype: int64

atlas_join
atlas_join
left_only     2910
both           244
right_only       0
Name: count, dtype: int64

ncsl_join
ncsl_join
both          3153
left_only        1
right_only       0
Name: count, dtype: int64

Duplicate county FIPS:
0


In [29]:
# Investigate missing CERF data

# Select CERF feature columns
cerf_features = [
    col for col in cerf_clean.columns
    if col != 'county_fips'
]

# Count missing values for each county
cerf_missing = cerf_clean.copy()

cerf_missing['missing_feature_count'] = (
    cerf_missing[cerf_features].isna().sum(axis=1)
)

print("Counties with at least one missing feature:")
print((cerf_missing['missing_feature_count'] > 0).sum())

print("\nCounties with all CERF features missing:")
print(
    (cerf_missing['missing_feature_count'] == len(cerf_features)).sum()
)

print("\nDistribution of missing features per county:")
display(
    cerf_missing['missing_feature_count']
    .value_counts()
    .sort_index()
)

print("\nSample records with missing features:")
display(
    cerf_missing.loc[
        cerf_missing['missing_feature_count'] > 0,
        ['county_fips', 'missing_feature_count']
    ].head(20)
)

Counties with at least one missing feature:
126

Counties with all CERF features missing:
0

Distribution of missing features per county:


missing_feature_count
0     3109
24     126
Name: count, dtype: int64


Sample records with missing features:


,county_fips,missing_feature_count
5,72085,24
75,02130,24
80,72115,24
81,78030,24
204,72105,24
242,02188,24
255,72053,24
259,72151,24
278,72067,24
317,02220,24


Cerf missing data issue why 126 on all rasters -There are three possibilities:
1. Raster coverage: Some counties or geographic areas may fall outside the valid coverage of the CERF raster files.
2. NoData values: Some raster cells may contain NoData values, which become NaN when aggregated.
3. County aggregation: Some county boundaries may not overlap valid raster cells, resulting in missing statistics.

Check	Result	Interpretation
CERF records	3,235	3,109 complete records and 126 with missing features
Missing CERF records	126	Each is missing the same 24 features
CERF joined to master	3,134	Successful FIPS matches
CERF unmatched in master	20	Counties without matching CERF records
Atlas joined to master	245	Counties with Atlas records
Atlas unmatched in master	2,909	No matching Atlas record
NCSL joined to master	3,153	Nearly complete legislative coverage
NCSL unmatched	1	One geographic record needs investigation
Duplicate county FIPS	0	No duplicate county keys

Why the CERF values are missing
Your missing county examples include FIPS codes beginning with:
- 72 — Puerto Rico
- 02 — Alaska
- 78 — U.S. Virgin Islands
That strongly suggests geographic coverage is contributing to the missing CERF values.